# MalPDF Vision ResNet-18 GPU handoff
Use the generated notebook from the export run, whose metadata checksum is already filled in. Upload the numeric ZIP as a **private** Kaggle Dataset and keep the Notebook private too. No raw PDF is needed. Enable Accelerator=GPU before running. This notebook uses one GPU (`cuda:0`) and never downloads pretrained weights or datasets.

The saved seed-42 split and four validation roles are preserved. Early stopping sees **base_selection only**, not the other validation roles. Test is forwarded once for probability export only; final Test metrics belong to the validation-locked stacking step back in Codespace. Runtime depends on the actual GPU and data I/O.

After dataset creation finishes, use **Add Input** to attach the dataset to this notebook. The bootstrap searches all of `/kaggle/input` for the matching sealed metadata, including renamed ZIPs or expanded bundles. If multiple matching copies are attached, set `INPUT_ROOT` to one dataset directory. Keep the expected checksum unchanged.

The performance configuration uses float32 RAM preload, two persistent Linux fork workers, pinned batches and CUDA FP16 AMP. RAM/shared-memory checks reserve working space. Batch size 256 is recorded before training; performance depends on the hardware. Matrices are not quantized or normalized again. Use this notebook before exporting Test probabilities; preserve previously completed output runs.


In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import hashlib, json, os, shutil, sys, zipfile

INPUT_ROOT = Path('/kaggle/input')
EXPECTED_METADATA_SHA256 = 'SET_FROM_EXPORT_REPORT'
CODE_OVERRIDES = {}
TRAINING_CODE_PROVENANCE = None
if len(EXPECTED_METADATA_SHA256) != 64:
    raise ValueError('Use the generated notebook, or copy metadata_sha256 from bundle_inventory.json')

# BEGIN BUNDLE DISCOVERY
def find_kaggle_bundle(input_root, expected_metadata_sha256):
    """Find an expanded bundle or ZIP by its pinned metadata, never its slug."""
    import hashlib
    import os
    from pathlib import Path
    import zipfile

    if len(expected_metadata_sha256) != 64 or any(c not in "0123456789abcdef" for c in expected_metadata_sha256):
        raise ValueError("Use the generated notebook with its pinned metadata checksum")
    root = Path(input_root)
    if not root.is_dir() or root.is_symlink():
        raise ValueError(f"INPUT_ROOT directory does not exist or is unsafe: {root}. "
                         "Finish dataset creation, then Add Input in this notebook; try Path('/kaggle/input').")
    boundary = root.resolve()
    expanded, archives, observed = [], [], []
    count = 0
    for directory, folders, files in os.walk(root, followlinks=False):
        folders[:] = sorted(f for f in folders if not (Path(directory) / f).is_symlink())
        count += len(folders) + len(files)
        if count > 200000:
            raise ValueError("Too many input entries; set INPUT_ROOT to this dataset's directory")
        for name in sorted(files):
            if name != "bundle_metadata.json" and Path(name).suffix.lower() != ".zip":
                continue
            path = Path(directory) / name
            if path.is_symlink() or not path.resolve().is_relative_to(boundary) or not path.is_file():
                raise ValueError("Unsafe bundle candidate path")
            if len(observed) < 10:
                observed.append(str(path.relative_to(root)))
            if name == "bundle_metadata.json":
                if path.stat().st_size > 1024 * 1024:
                    raise ValueError("Bundle metadata exceeds its cap")
                with path.open("rb") as handle:
                    content = handle.read(1024 * 1024 + 1)
                if hashlib.sha256(content).hexdigest() == expected_metadata_sha256:
                    expanded.append(path.parent)
                continue
            if path.stat().st_size > 7 * 1024**3:
                continue
            try:
                with zipfile.ZipFile(path) as archive:
                    members = archive.infolist()
                    if len(members) > 22000 or sum(m.file_size for m in members) > 7 * 1024**3:
                        continue
                    metadata = [m for m in members if m.filename == "bundle_metadata.json"]
                    if len(metadata) != 1:
                        continue
                    member = metadata[0]
                    if (member.file_size > 1024 * 1024 or member.flag_bits & 1
                            or member.compress_type not in (zipfile.ZIP_STORED, zipfile.ZIP_DEFLATED)
                            or (member.external_attr >> 16 & 0o170000) == 0o120000):
                        continue
                    with archive.open(member) as handle:
                        content = handle.read(1024 * 1024 + 1)
                    if hashlib.sha256(content).hexdigest() == expected_metadata_sha256:
                        archives.append(path)
            except (zipfile.BadZipFile, EOFError, RuntimeError, NotImplementedError):
                continue
    # Some mounts contain both the original ZIP and its expanded contents.
    matches = expanded if expanded else archives
    if len(matches) > 1:
        raise ValueError("Ambiguous matching bundles; set INPUT_ROOT to one dataset directory. "
                         f"Matches: {[str(p) for p in matches[:10]]}")
    if not matches:
        mounted = sorted(p.name for p in root.iterdir())[:10]
        raise ValueError(f"No bundle matches the pinned Codespace export under {root}. "
                         f"Attached entries: {mounted}; metadata/ZIP candidates: {observed}. "
                         "Wait for dataset creation, then Add Input: nguyenkhang294/malpdf-markov-cache. "
                         "Use the generated notebook for that export; keep its checksum unchanged.")
    return (matches[0], None) if expanded else (None, matches[0])

def prepare_kaggle_code(bundle_root, metadata, overrides):
    """Copy verified base code into a new directory with pinned source updates."""
    import hashlib
    from pathlib import Path, PurePosixPath
    import tempfile

    if not overrides:
        return Path(bundle_root) / 'code/src'
    allowed = {'code/src/malpdf/experiments/kaggle_vision_runner.py',
               'code/src/malpdf/deep_learning/training.py'}
    if set(overrides) != allowed or not allowed.issubset(metadata['file_sha256']):
        raise ValueError('Unexpected optimized code overlay')
    for item in overrides.values():
        content = item['source'].encode('utf-8')
        if len(content) > 1024 * 1024 or hashlib.sha256(content).hexdigest() != item['sha256']:
            raise ValueError('Optimized source checksum mismatch')
    destination = Path(tempfile.mkdtemp(prefix='malpdf_gpu_code_'))
    for name, digest in metadata['file_sha256'].items():
        if not name.startswith('code/src/'):
            continue
        member = PurePosixPath(name)
        source = Path(bundle_root).joinpath(*member.parts)
        if (member.is_absolute() or '..' in member.parts or '\\' in name or member.suffix != '.py'
                or source.is_symlink() or not source.resolve().is_relative_to(Path(bundle_root).resolve())):
            raise ValueError('Unsafe sealed code path')
        with source.open('rb') as handle:
            content = handle.read(1024 * 1024 + 1)
        if len(content) > 1024 * 1024 or hashlib.sha256(content).hexdigest() != digest:
            raise ValueError('Base source checksum mismatch')
        if name in overrides:
            content = overrides[name]['source'].encode('utf-8')
        target = destination.joinpath(*member.parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open('xb') as handle:
            handle.write(content)
    return destination / 'code/src'
# END BUNDLE DISCOVERY

BUNDLE_ROOT, bundle_archive = find_kaggle_bundle(INPUT_ROOT, EXPECTED_METADATA_SHA256)
if bundle_archive is not None:
    BUNDLE_ROOT = Path('/tmp') / ('malpdf_markov_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    BUNDLE_ROOT.mkdir(exist_ok=False)
    with zipfile.ZipFile(bundle_archive) as archive:
        members = archive.infolist()
        if len(members) > 22000 or sum(m.file_size for m in members) > 7 * 1024**3:
            raise ValueError('Archive exceeds the numeric bundle caps')
        seen = set()
        for member in members:
            name = PurePosixPath(member.filename)
            mode = member.external_attr >> 16
            if (member.filename in seen or name.is_absolute() or '..' in name.parts
                    or '\\' in member.filename or (mode & 0o170000) == 0o120000
                    or member.is_dir() or member.compress_type not in (zipfile.ZIP_STORED, zipfile.ZIP_DEFLATED)
                    or name.suffix not in ('.npy', '.py', '.csv', '.json')):
                raise ValueError('Unsafe or unexpected archive member')
            seen.add(member.filename)
            cap = 16 * 1024**2 if name.suffix == '.csv' else 1024 * 1024
            if member.file_size > cap:
                raise ValueError('Archive member exceeds its cap')
            target = BUNDLE_ROOT.joinpath(*name.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            remaining = member.file_size
            with archive.open(member) as source, target.open('xb') as output:
                while remaining:
                    block = source.read(min(64 * 1024, remaining))
                    if not block:
                        raise ValueError('Truncated archive member')
                    output.write(block)
                    remaining -= len(block)
                if source.read(1):
                    raise ValueError('Archive member grew beyond its declaration')

metadata_bytes = (BUNDLE_ROOT / 'bundle_metadata.json').read_bytes()
if hashlib.sha256(metadata_bytes).hexdigest() != EXPECTED_METADATA_SHA256:
    raise ValueError('Dataset metadata differs from the sealed Codespace export')
metadata = json.loads(metadata_bytes)
for name, digest in metadata['file_sha256'].items():
    member = PurePosixPath(name)
    path = BUNDLE_ROOT.joinpath(*member.parts)
    if member.is_absolute() or '..' in member.parts or not path.resolve().is_relative_to(BUNDLE_ROOT.resolve()) or path.is_symlink():
        raise ValueError('Unsafe metadata path')
    if path.stat().st_size > 16 * 1024**2 or hashlib.sha256(path.read_bytes()).hexdigest() != digest:
        raise ValueError('Code/manifest/config checksum mismatch')
MODEL_CODE_ROOT = prepare_kaggle_code(BUNDLE_ROOT, metadata, CODE_OVERRIDES)
sys.dont_write_bytecode = True
sys.path.insert(0, str(MODEL_CODE_ROOT))
import malpdf
if Path(malpdf.__file__).resolve().parent != (MODEL_CODE_ROOT / 'malpdf').resolve():
    raise ValueError('Restart kernel to load the sealed package')
print({'partition_rows': metadata['partition_rows'], 'code_commit': metadata['code_commit'], 'raw_pdfs': False})


In [ ]:
import torch
from malpdf.experiments.kaggle_vision_runner import load_bundle, train_kaggle_vision

if not torch.cuda.is_available():
    raise RuntimeError('Enable Accelerator=GPU in Kaggle settings before training')
frame, roles, bundle = load_bundle(BUNDLE_ROOT)
configuration = json.loads((BUNDLE_ROOT / 'vision_configuration.json').read_text())
configuration.update(batch_size=256, preload_ram=True, mixed_precision=True, num_workers=2, prefetch_factor=1, max_ram_cache_gib=6)
print({'gpu': torch.cuda.get_device_name(0), 'configuration': configuration, 'role_counts': roles.role.value_counts().to_dict()})


In [ ]:
OUTPUT_DIR = train_kaggle_vision(
    BUNDLE_ROOT,
    Path('/kaggle/working/vision_outputs'),
    configuration,
    code_provenance=TRAINING_CODE_PROVENANCE,
)


Download the entire new output run directory, including checksums and selection lock. It contains `vision_resnet_best.pt`, `vision_probs_val.csv`, `vision_probs_test.csv` and separate `vision_probs_unlabeled.csv` for the four unknown samples. Do not edit IDs or merge CSVs by row position. The private Codespace `private_id_map.csv` maps opaque IDs back to the original locked sample IDs. Preserve all 25 tabular fold checkpoints; they can be reused for the later stacking continuation.


In [ ]:
print(OUTPUT_DIR)
print(json.loads((OUTPUT_DIR / 'output_checksums.json').read_text()))
